# Directional Network Simulation Walkthrough

This notebook explores the Virar-Dadar double-line model. Run it from top to bottom to inspect UP/DOWN lines, directional block occupancy, same-direction loop overtaking, history, and metrics. The final sections are deliberately editable so new scenarios can be tried without changing the core files.

In [1]:
import subprocess
import sys
from pathlib import Path

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'scenarios').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root.')

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from models.train import Train
from scenarios.scenario_1 import build_trains
from src.planning.actions import ActionType
from src.planning.scheduler import Scheduler
from src.railway.network import DOWN, UP, default_network
from src.railway.occupancy import OccupancyState
from src.simulation.simulator import Simulator

repo_root


WindowsPath('C:/Users/mikhi/OneDrive/Desktop/Incomplete Projects/Train_Traffic_Control')

## 1. Inspect the topology

Increasing station indexes are UP in this model: Virar toward Dadar and onward toward Churchgate. Decreasing indexes are DOWN. Every station has two mains; Borivali and Andheri have separate loops for both directions.

In [3]:
topology = [
    {
        'index': index,
        'station': station.name,
        'up_main': True,
        'down_main': True,
        'up_loop': station.has_loop(UP),
        'down_loop': station.has_loop(DOWN),
    }
    for index, station in enumerate(default_network.stations)
]

topology


[{'index': 0,
  'station': 'Virar',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False},
 {'index': 1,
  'station': 'Bhayandar',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False},
 {'index': 2,
  'station': 'Borivali',
  'up_main': True,
  'down_main': True,
  'up_loop': True,
  'down_loop': True},
 {'index': 3,
  'station': 'Andheri',
  'up_main': True,
  'down_main': True,
  'up_loop': True,
  'down_loop': True},
 {'index': 4,
  'station': 'Bandra',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False},
 {'index': 5,
  'station': 'Dadar',
  'up_main': True,
  'down_main': True,
  'up_loop': False,
  'down_loop': False}]

## 2. Compare parallel directional blocks

The station pair is the same, but direction is part of the block key. Reserving Borivali-Andheri UP must not reserve Borivali-Andheri DOWN.

In [12]:
up_block = default_network.block_between(2, 3)
down_block = default_network.block_between(3, 2)
block_state = OccupancyState(default_network)
block_state.reserve_block(up_block.key, 'TRAIN-UP_DEMO', 2, 3)
down_available, down_reason = block_state.can_reserve_block(
    down_block.key, 'TRAIN-DOWN_DEMO', 3, 2
)

{
    'up_block': up_block.key,
    'down_block': down_block.key,
    'keys_are_distinct': up_block.key != down_block.key,
    'down_available_after_up_reservation': down_available,
    'reason': down_reason,
}


{'up_block': (2, 3, 'up'),
 'down_block': (2, 3, 'down'),
 'keys_are_distinct': True,
 'down_available_after_up_reservation': True,
 'reason': 'directional block available'}

In [8]:
print(up_block.key)

(2, 3, 'up')


## 3. Run the default mixed-direction scenario

`UP_EXP1` starts immediately behind `UP_LOC1` at loop-equipped Borivali. The scheduler should place the lower-priority local in `up_loop`, move the express through `up_main`, and move `DOWN_LOC1` independently on `down_main`.

In [14]:
def train_table(simulator):
    return [
        {
            'name': train.name,
            'direction': train.line.split('_', 1)[0].upper(),
            'station': simulator.network.station_name(train.current_station),
            'line': train.line,
            'destination': simulator.network.station_name(train.destination_station),
            'priority': train.priority,
            'finished': train.finished,
            'waiting_time': train.waiting_time,
            'loop_entries': train.loop_entries,
        }
        for train in simulator.trains
    ]

def action_table(actions):
    return [
        {
            'train': action.train_name,
            'action': action.action_type.value,
            'source_line': action.source_track,
            'target_line': action.target_track,
            'block': action.block,
            'reason': action.reason,
            'conflict': action.conflict,
        }
        for action in actions
    ]

def occupancy_table(state):
    return state.snapshot()['stations']


In [13]:
trains = build_trains()
sim = Simulator(trains, Scheduler(), network=default_network, verbose=False)

{
    'trains': train_table(sim),
    'occupancy': occupancy_table(sim.occupancy_state),
}


{'trains': [{'name': 'UP_EXP1',
   'direction': 'UP',
   'station': 'Bhayandar',
   'line': 'up_main',
   'destination': 'Dadar',
   'priority': 10,
   'finished': False,
   'waiting_time': 0,
   'loop_entries': 0},
  {'name': 'UP_LOC1',
   'direction': 'UP',
   'station': 'Borivali',
   'line': 'up_main',
   'destination': 'Dadar',
   'priority': 1,
   'finished': False,
   'waiting_time': 0,
   'loop_entries': 0},
  {'name': 'DOWN_LOC1',
   'direction': 'DOWN',
   'station': 'Dadar',
   'line': 'down_main',
   'destination': 'Virar',
   'priority': 1,
   'finished': False,
   'waiting_time': 0,
   'loop_entries': 0}],
 'occupancy': [{'station': 'Virar', 'up_main': None, 'down_main': None},
  {'station': 'Bhayandar', 'up_main': 'UP_EXP1', 'down_main': None},
  {'station': 'Borivali',
   'up_main': 'UP_LOC1',
   'down_main': None,
   'up_loop': None,
   'down_loop': None},
  {'station': 'Andheri',
   'up_main': None,
   'down_main': None,
   'up_loop': None,
   'down_loop': None},
  {'

In [15]:
first_actions = sim.step()

{
    'actions': action_table(first_actions),
    'trains_after_tick': train_table(sim),
    'occupancy_after_tick': sim.history[-1]['occupancy']['stations'],
    'directional_blocks_used': sim.history[-1]['occupancy']['blocks'],
}


{'actions': [{'train': 'UP_LOC1',
   'action': 'ENTER_LOOP',
   'source_line': 'up_main',
   'target_line': 'up_loop',
   'block': None,
   'reason': 'enter up_loop at Borivali to allow UP_EXP1 to overtake',
   'conflict': False},
  {'train': 'UP_EXP1',
   'action': 'MOVE',
   'source_line': 'up_main',
   'target_line': 'up_main',
   'block': (1, 2, 'up'),
   'reason': 'advance toward destination',
   'conflict': False},
  {'train': 'DOWN_LOC1',
   'action': 'MOVE',
   'source_line': 'down_main',
   'target_line': 'down_main',
   'block': (4, 5, 'down'),
   'reason': 'advance toward destination',
   'conflict': False}],
 'trains_after_tick': [{'name': 'UP_EXP1',
   'direction': 'UP',
   'station': 'Borivali',
   'line': 'up_main',
   'destination': 'Dadar',
   'priority': 10,
   'finished': False,
   'waiting_time': 0,
   'loop_entries': 0},
  {'name': 'UP_LOC1',
   'direction': 'UP',
   'station': 'Borivali',
   'line': 'up_loop',
   'destination': 'Dadar',
   'priority': 1,
   'finis

In [16]:
first_by_train = {action.train_name: action for action in first_actions}
assert first_by_train['UP_LOC1'].action_type == ActionType.ENTER_LOOP
assert first_by_train['UP_EXP1'].action_type == ActionType.MOVE
assert first_by_train['DOWN_LOC1'].action_type == ActionType.MOVE

'Tick 0 behaved as expected: UP overtaking and DOWN movement happened safely.'


'Tick 0 behaved as expected: UP overtaking and DOWN movement happened safely.'

In [17]:
while sim.active_trains():
    sim.step()

sim.get_metrics()


{'total_ticks': 5,
 'arrived_trains': 3,
 'active_trains': 0,
 'throughput': 0.6,
 'conflict_count': 0,
 'loop_usage': 1,
 'trains': {'UP_EXP1': {'finished': True,
   'waiting_time': 0,
   'completion_time': 4,
   'current_station': 'Dadar',
   'line': 'up_main',
   'loop_entries': 0},
  'UP_LOC1': {'finished': True,
   'waiting_time': 0,
   'completion_time': 5,
   'current_station': 'Dadar',
   'line': 'up_main',
   'loop_entries': 1},
  'DOWN_LOC1': {'finished': True,
   'waiting_time': 0,
   'completion_time': 5,
   'current_station': 'Virar',
   'line': 'down_main',
   'loop_entries': 0}}}

In [18]:
timeline = [
    {
        'time': tick['time'],
        **action_row,
    }
    for tick in sim.history
    for action_row in action_table(tick['actions'])
]

timeline


[{'time': 0,
  'train': 'UP_LOC1',
  'action': 'ENTER_LOOP',
  'source_line': 'up_main',
  'target_line': 'up_loop',
  'block': None,
  'reason': 'enter up_loop at Borivali to allow UP_EXP1 to overtake',
  'conflict': False},
 {'time': 0,
  'train': 'UP_EXP1',
  'action': 'MOVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (1, 2, 'up'),
  'reason': 'advance toward destination',
  'conflict': False},
 {'time': 0,
  'train': 'DOWN_LOC1',
  'action': 'MOVE',
  'source_line': 'down_main',
  'target_line': 'down_main',
  'block': (4, 5, 'down'),
  'reason': 'advance toward destination',
  'conflict': False},
 {'time': 1,
  'train': 'UP_EXP1',
  'action': 'MOVE',
  'source_line': 'up_main',
  'target_line': 'up_main',
  'block': (2, 3, 'up'),
  'reason': 'advance toward destination',
  'conflict': False},
 {'time': 1,
  'train': 'DOWN_LOC1',
  'action': 'MOVE',
  'source_line': 'down_main',
  'target_line': 'down_main',
  'block': (3, 4, 'down'),
  'reason': 'advance t

## 4. Mirror the overtake in the DOWN direction

This checks that the implementation is symmetric: a lower-priority DOWN local at Borivali should enter `down_loop` for a DOWN express approaching from Andheri.

In [10]:
down_trains = [
    Train('DOWN_EXP', 'EXPRESS', 10, current_station=3, destination_station=0),
    Train('DOWN_LOC', 'LOCAL', 1, current_station=2, destination_station=0),
]
down_sim = Simulator(
    down_trains, Scheduler(), network=default_network, verbose=False
)
down_actions = down_sim.step()

{
    'actions': action_table(down_actions),
    'trains_after_tick': train_table(down_sim),
}


{'actions': [{'train': 'DOWN_LOC',
   'action': 'ENTER_LOOP',
   'source_line': 'down_main',
   'target_line': 'down_loop',
   'block': None,
   'reason': 'enter down_loop at Borivali to allow DOWN_EXP to overtake',
   'conflict': False},
  {'train': 'DOWN_EXP',
   'action': 'MOVE',
   'source_line': 'down_main',
   'target_line': 'down_main',
   'block': (2, 3, 'down'),
   'reason': 'advance toward destination',
   'conflict': False}],
 'trains_after_tick': [{'name': 'DOWN_EXP',
   'direction': 'DOWN',
   'station': 'Borivali',
   'line': 'down_main',
   'destination': 'Virar',
   'priority': 10,
   'finished': False,
   'waiting_time': 0,
   'loop_entries': 0},
  {'name': 'DOWN_LOC',
   'direction': 'DOWN',
   'station': 'Borivali',
   'line': 'down_loop',
   'destination': 'Virar',
   'priority': 1,
   'finished': False,
   'waiting_time': 0,
   'loop_entries': 1}]}

## 5. Editable experiment

Change train positions, destinations, priorities, or types below. Keep active trains on different station/directional-line resources at initialization. With the current one-station-per-tick model, begin a high-priority train directly behind a lower-priority train at Borivali or Andheri to observe an overtake.

In [11]:
experiment_trains = [
    Train('MY_EXP', 'EXPRESS', 10, current_station=1, destination_station=5),
    Train('MY_LOCAL', 'LOCAL', 1, current_station=2, destination_station=5),
    Train('MY_DOWN', 'LOCAL', 1, current_station=5, destination_station=0),
]
experiment = Simulator(
    experiment_trains, Scheduler(), network=default_network, verbose=False
)
experiment_metrics = experiment.run(max_ticks=30)

{
    'metrics': experiment_metrics,
    'timeline': [
        {'time': tick['time'], **row}
        for tick in experiment.history
        for row in action_table(tick['actions'])
    ],
}


{'metrics': {'total_ticks': 5,
  'arrived_trains': 3,
  'active_trains': 0,
  'throughput': 0.6,
  'conflict_count': 0,
  'loop_usage': 1,
  'trains': {'MY_EXP': {'finished': True,
    'waiting_time': 0,
    'completion_time': 4,
    'current_station': 'Dadar',
    'line': 'up_main',
    'loop_entries': 0},
   'MY_LOCAL': {'finished': True,
    'waiting_time': 0,
    'completion_time': 5,
    'current_station': 'Dadar',
    'line': 'up_main',
    'loop_entries': 1},
   'MY_DOWN': {'finished': True,
    'waiting_time': 0,
    'completion_time': 5,
    'current_station': 'Virar',
    'line': 'down_main',
    'loop_entries': 0}}},
 'timeline': [{'time': 0,
   'train': 'MY_LOCAL',
   'action': 'ENTER_LOOP',
   'source_line': 'up_main',
   'target_line': 'up_loop',
   'block': None,
   'reason': 'enter up_loop at Borivali to allow MY_EXP to overtake',
   'conflict': False},
  {'time': 0,
   'train': 'MY_EXP',
   'action': 'MOVE',
   'source_line': 'up_main',
   'target_line': 'up_main',
   

## 6. Run the core test suite

Use this after experimenting with core files. A nonzero return code fails the cell and prints the test output.

In [12]:
test_run = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-v'],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(test_run.stdout)
print(test_run.stderr)
assert test_run.returncode == 0, 'Core tests failed.'
'All core tests passed.'



test_default_mixed_direction_scenario_completes_safely (tests.test_core_engine.CoreEngineTest.test_default_mixed_direction_scenario_completes_safely) ... ok
test_default_network_maps_virar_to_dadar_as_up (tests.test_core_engine.CoreEngineTest.test_default_network_maps_virar_to_dadar_as_up) ... ok
test_directional_loops_exist_only_where_configured (tests.test_core_engine.CoreEngineTest.test_directional_loops_exist_only_where_configured) ... ok
test_higher_priority_down_train_overtakes_through_down_loop (tests.test_core_engine.CoreEngineTest.test_higher_priority_down_train_overtakes_through_down_loop) ... ok
test_higher_priority_up_train_overtakes_through_up_loop (tests.test_core_engine.CoreEngineTest.test_higher_priority_up_train_overtakes_through_up_loop) ... ok
test_movement_rejects_a_block_from_the_wrong_direction (tests.test_core_engine.CoreEngineTest.test_movement_rejects_a_block_from_the_wrong_direction) ... ok
test_no_directional_loop_causes_an_explained_wait (tests.test_core_en

'All core tests passed.'

In [ ]:
# The scheduler then performs this logic in [scheduler.py (line 89)](C:/Users/mikhi/OneDrive/Desktop/Incomplete Projects/Train_Traffic_Control/src/planning/scheduler.py:89):
# Examine the higher-priority express.
# Find its next station, Borivali.
# Notice that the Borivali up_main is currently occupied by UP_LOC1.
# Confirm that the local has lower priority.
# Confirm both trains travel in the same direction.
# Confirm Borivali has an empty UP loop.
# Mark the local as forced to enter the loop.